# Phase 6 — Qwen2.5-1.5B + LoRA. Same data and eval protocol as Phase 5 (ModernBERT-base).

> **Legacy notebook.** This is the first-round Colab version (legacy dataset: the 7,580 rows of `data/derived/`, evaluated on the 102-row `test_real.jsonl`). The final models (ModernBERT v3b, test AUROC 0.771; Qwen2.5-1.5B, 0.765) were trained with the Kaggle versions of the notebooks, `kaggle_phase5_modernbert.ipynb` and `kaggle_phase6_qwen.ipynb` (generated from this notebook by `make_kaggle_notebooks.py`), with `DATASET = "final"` on the v3b data (7,154 training rows, `REAL_WEIGHT = 3`, policy block on, 4,096 tokens, one T4). The rest of this header describes the legacy setup. See `docs/kaggle_runbook.md`.

Binary classifier: given a conversation and a **candidate tool call**, predict `incorrect` (class 1, the positive class) or `correct` (class 0). `Qwen/Qwen2.5-1.5B` with a 2-way sequence-classification head, 4-bit quantized (QLoRA) and fine-tuned with LoRA so it fits a **Colab T4 (16 GB)**. The input text, split, metrics and evaluation are the Phase 5 code, copied.

**Before running (Colab, GPU runtime = T4):**
1. Copy the repo's `data/derived/` folder to `MyDrive/toolcall-verifier/data/derived/`. Needed: `train_pool`, `bfcl_pool`, `positives_class2_controls`, `positives_class4_controls`, `positives_class6_controls`, `negatives_pool_final` (3,029 rows, classes 1-6) and `test_real`.
2. Run Cell 1, then **Runtime > Restart session** if Colab asks, then run the cells top to bottom.

Checkpoints go to `MyDrive/toolcall-verifier/checkpoints/phase6/`. Re-runnable: an existing `best/` adapter skips training, and an interrupted run resumes from its last epoch checkpoint (a checkpoint is the small LoRA adapter plus optimizer state, so saving to Drive is cheap).

**Expect several hours on a T4** (a rough estimate, not measured: roughly an hour or more per epoch). Checkpoints are saved once per epoch, so a Colab disconnect can cost up to an epoch.

**Differences from the spec, all needed to make it run on a T4 / current libraries** (details in the cell comments):
- `warmup_steps` (10% of total steps) instead of `warmup_ratio=0.1`: current `transformers` no longer accepts `warmup_ratio`.
- Gradient checkpointing and `prepare_model_for_kbit_training` (standard for QLoRA): 8 x 1536 tokens does not fit in 16 GB without them.
- A `BitsAndBytesConfig` instead of the bare `load_in_4bit=True` argument (the supported form), the model dtype forced to fp16 (the default would load Qwen's bf16 weights, which a T4 does not support), and trainable weights kept in fp32 so fp16 training works.
- The cell starts at micro-batch 2 (`MICRO_BATCH`) with gradient checkpointing on and retries at 1 if it runs out of memory, keeping the effective batch at 16 (D-029).
- The legacy training data has no Class 7 (synthetic Class 7 examples were removed, D-009); `test_real` has 8 real Class 7 rows, so expect near-zero recall on them. The final dataset has Class 7 (state flips and real errors, D-021, D-026, D-028).

## Cell 1: install and imports

In [ ]:
# --- final mode (D-023) ---
DATASET = "legacy"  # "legacy": the 7,580-row dataset of D-012 with the frozen split. "final": the builder's output (D-022): split taken from each row, INCLUDE_POLICY on
INCLUDE_POLICY_OVERRIDE = None  # None: the POLICY block follows DATASET ("final": on). True / False forces it, e.g. False to score a model that was trained without the block (D-023)
BENCHMARK_STEPS = None  # e.g. 100: train that many optimizer steps, print seconds per step and the projected time of a full run, then stop WITHOUT saving (about 15 minutes on a T4)
MICRO_BATCH = 2                # per-device micro-batch the training starts with (16 must divide by it). The effective batch stays 16: accumulation = 16 / MICRO_BATCH. The retry ladder only goes DOWN from here (2 -> 1)
GRADIENT_CHECKPOINTING = True  # True (default): recompute activations in the backward pass; needed for 4,096-token Qwen inputs (about 14 GB of activations per row without it). False only for short inputs; None counts as True
# COLAB: pip install ...
!pip install -q transformers peft accelerate bitsandbytes datasets scikit-learn

import gc, json, math, os, random, re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import datasets
from datasets import Dataset
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training

## Cell 2: mount Drive and load the data

The six training files are combined exactly as in Phase 5 (positives are label 0 = `correct`, negatives are label 1 = `incorrect`); the row count is asserted. `test_real.jsonl` (102 rows) is evaluation-only and is not part of the 7,580 legacy training rows; it is loaded here so a missing file fails now, not after hours of training.

In [ ]:
import gc, json, os, random, re
from collections import Counter
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/toolcall-verifier")
except ImportError:  # not on Colab: use a local checkout, or set TCV_ROOT
    ROOT = Path(os.environ.get("TCV_ROOT", "."))

DATA = ROOT / "data" / "derived"
CKPT_DIR = ROOT / "checkpoints" / "phase6"
BEST_DIR = CKPT_DIR / "best"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SMOKE = os.environ.get("TCV_SMOKE") == "1"  # local CPU smoke test only; leave unset on Colab


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


if DATASET == "legacy":
    # file -> the label every row in it must have
    TRAIN_FILES = {
        "train_pool.jsonl": "correct",
        "bfcl_pool.jsonl": "correct",
        "positives_class2_controls.jsonl": "correct",
        "positives_class4_controls.jsonl": "correct",
        "positives_class6_controls.jsonl": "correct",
        "negatives_pool_final.jsonl": "incorrect",
    }
    rows = []
    for name, expected in TRAIN_FILES.items():
        part = load_jsonl(DATA / name)
        wrong = [r["trajectory_id"] for r in part if r["label"] != expected]
        assert not wrong, f"{name}: {len(wrong)} rows are not label={expected!r}"
        for r in part:
            r["_file"] = name
        rows += part
        print(f"{name:34s} {len(part):5d} rows  (label={expected})")

    assert len({r["trajectory_id"] for r in rows}) == len(rows), "duplicate trajectory_ids"
    EXPECTED_ROWS = 7580  # the six training files; test_real.jsonl is evaluation-only and not counted
    assert len(rows) == EXPECTED_ROWS, f"expected {EXPECTED_ROWS} training rows, found {len(rows)}: copy the current data/derived/ to Drive"
    test_rows = load_jsonl(DATA / "test_real.jsonl")
    assert len(test_rows) == 102, f"expected 102 test_real rows, found {len(test_rows)}"
    print(f"test_real.jsonl {len(test_rows):5d} rows  (evaluation only)")
else:
    assert DATASET == "final", f'DATASET must be "legacy" or "final", not {DATASET!r}'

    def final_path(name):  # data/derived/final/NAME.jsonl, or final_NAME.jsonl next to the other data files (the Kaggle upload); TCV_FINAL_DIR overrides
        for d in (os.environ.get("TCV_FINAL_DIR"), DATA / "final", DATA):
            for fn in (f"{name}.jsonl", f"final_{name}.jsonl"):
                if d and (Path(d) / fn).exists():
                    return Path(d) / fn
        raise FileNotFoundError(f"final dataset file {name}.jsonl not found in {DATA}: run data/package_for_kaggle.py and upload it, or set DATASET = \"legacy\"")

    FINAL = {s: load_jsonl(final_path(s)) for s in ("train", "val", "test", "test_synth")}
    for part in FINAL.values():
        for r in part:
            r["_file"] = "train_pool.jsonl" if r.get("source") == "tau2_gold" else str(r.get("source"))
    rows = FINAL["train"] + FINAL["val"]
    test_rows = FINAL["test"]
    # the only failures: an empty split, or a synthetic row in the test split
    assert all(FINAL.values()), f"empty split(s): {[s for s, p in FINAL.items() if not p]}"
    bad = [r["trajectory_id"] for r in FINAL["test"] if r.get("origin") == "synthetic" or str(r.get("source", "")).startswith("synthetic")]
    assert not bad, f"synthetic rows in the test split: {bad[:5]}"
    ids = [r["trajectory_id"] for p in FINAL.values() for r in p]
    off = [s for s, p in FINAL.items() for r in p if r.get("split") not in (s, None)]
    print(f"final dataset from {final_path('train').parent}: {len(ids)} rows" + ("" if len(ids) == len(set(ids)) else f" (WARNING: {len(ids) - len(set(ids))} duplicate ids)")
          + ("" if not off else f" (WARNING: {len(off)} rows whose split field differs from their file)"))

    def final_counts(part_by_split):
        def cls(r):
            return f"class {r['error_class']}" if r["label"] == "incorrect" else "correct"
        for title, key in (("class", cls), ("source", lambda r: r.get("source")), ("origin", lambda r: r.get("origin")), ("domain", lambda r: r["domain"])):
            cnt = Counter((str(key(r)), s) for s, p in part_by_split.items() for r in p)
            names = sorted({n for n, _ in cnt})
            print(f"\nrows per {title} and split:")
            print(f"   {'':30s}" + "".join(f"{s:>11s}" for s in part_by_split) + f"{'total':>9s}")
            for n in names:
                vals = [cnt[(n, s)] for s in part_by_split]
                print(f"   {n:30s}" + "".join(f"{v:>11d}" for v in vals) + f"{sum(vals):>9d}")

    final_counts(FINAL)

LABEL2ID = {"correct": 0, "incorrect": 1}  # "incorrect" is the positive class everywhere
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print(f"\ntotal: {len(rows)} examples")
print("label balance:", dict(Counter(r["label"] for r in rows)))
print("incorrect by error class:")
for (c, n), k in sorted({(r["error_class"], r["error_class_name"]): 0 for r in rows if r["label"] == "incorrect"}.items()):
    print(f"  class {c} ({n}): {sum(1 for r in rows if r['label'] == 'incorrect' and r['error_class'] == c)}")
print("domains:", dict(Counter(r["domain"] for r in rows)))

print("\n3-row sample:")
for r in random.Random(SEED).sample(rows, 3):
    call = json.dumps(r["candidate_tool_call"], ensure_ascii=False)
    print(f"  {r['trajectory_id'][:70]}")
    print(f"    domain={r['domain']} label={r['label']} error_class={r.get('error_class')} turns={len(r['turns'])} file={r['_file']}")
    print(f"    candidate: {call[:140]}")

## Cell 3: `format_input` and the train/val split

`format_input` and the split are the Phase 5 code, unchanged except for the tokenizer: the token-budget guard now counts **Qwen** tokens. Tool schema = names + parameter names only; last 10 turns, tool results cut to 200 chars; an `EARLIER_CALLS:` line for calls before that window; the candidate call last. The split is `StratifiedGroupKFold` (10 folds, first fold as validation, seed 42) grouped by `(domain, task_id)`, so a negative and its control always land in the same split. The validation groups are frozen in the split cell (6,816 train / 764 val rows, asserted exactly) because the fold `StratifiedGroupKFold` returns depends on the scikit-learn version (6,892 / 688 up to 1.7, 6,816 / 764 from 1.8).

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LEN = 1536
TOKEN_BUDGET = 1500
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Qwen2.5 pads with its EOS token; set explicitly
tokenizer.padding_side = "right"

# tightening ladder: first level is the spec; later levels only fire when over budget
LEVELS = [
    dict(n_turns=10, tool_chars=200, msg_chars=500, earlier=20, arg_chars=100),
    dict(n_turns=10, tool_chars=100, msg_chars=300, earlier=10, arg_chars=60),
    dict(n_turns=8, tool_chars=60, msg_chars=200, earlier=5, arg_chars=40),
    dict(n_turns=6, tool_chars=40, msg_chars=150, earlier=3, arg_chars=30),
]
CAND_ARG_CHARS = 300  # candidate args are shown almost in full: the error is often in one value


def _clip(s, n):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= n else s[: n - 1] + "…"


def _val(v, n):
    return _clip(v if isinstance(v, str) else json.dumps(v, ensure_ascii=False), n)


def _call(name, args, n):
    return f"{name}(" + ", ".join(f"{k}={_val(v, n)}" for k, v in (args or {}).items()) + ")"


def _split_top(s):
    parts, depth, cur = [], 0, ""
    for ch in s:
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append(cur)
            cur = ""
        else:
            cur += ch
    if cur.strip():
        parts.append(cur)
    return parts


_schema_cache = {}
_requires_cache = {}


def parse_requires(text):
    # {tool_name: "prerequisite rule"} from the "  # Requires: ..." line that follows a tool in
    # the tau2-bench schema text (data/tool_schemas.py PREREQUISITES). BFCL schemas have none.
    if text in _requires_cache:
        return _requires_cache[text]
    rules, cur = {}, None
    for line in text.splitlines():
        if line.startswith("- ") and "(" in line:
            cur = line[2 : line.index("(")].strip()
        elif cur and line.startswith("  # Requires:"):
            rules[cur] = line[len("  # Requires:"):].strip()
    _requires_cache[text] = rules
    return rules



def parse_schema(text):
    # {tool_name: [param names]} from either schema text format in our data:
    #   tau2-bench: "- name(p: type, q: type (optional)): description"
    #   BFCL:       "Function: name" / "  Parameters:" / "    p (type) [required]: description"
    if text in _schema_cache:
        return _schema_cache[text]
    tools, cur = {}, None
    for line in text.splitlines():
        if line.startswith("- ") and "(" in line:
            name = line[2 : line.index("(")].strip()
            depth, end = 0, None
            for i in range(line.index("("), len(line)):
                depth += (line[i] == "(") - (line[i] == ")")
                if depth == 0:
                    end = i
                    break
            inner = line[line.index("(") + 1 : end] if end else ""
            tools[name] = [p.split(":")[0].strip() for p in _split_top(inner) if p.strip()]
        elif line.startswith("Function: "):
            cur = line[len("Function: "):].strip()
            tools[cur] = []
        elif cur and re.match(r"^    \w+ \(", line):
            tools[cur].append(line.split("(")[0].strip())
    _schema_cache[text] = tools
    return tools


def _format(ex, n_turns, tool_chars, msg_chars, earlier, arg_chars):
    turns = ex["turns"]
    cand = ex["candidate_tool_call"]
    start = max(0, len(turns) - n_turns)
    window = turns[start:]

    # 1. tools: called in the last 5 turns, plus the candidate's tool
    schema = parse_schema(ex.get("tool_schemas_text", ""))
    wanted = []
    for t in turns[-5:]:
        if t.get("role") == "assistant_tool_call" and t["name"] not in wanted:
            wanted.append(t["name"])
    if cand["name"] not in wanted:
        wanted.append(cand["name"])
    tool_strs = [f"{n}({', '.join(schema[n])})" for n in wanted if n in schema]
    # the prerequisite rule is shown for the candidate's tool only: it is what the call is judged
    # against, and rules for every listed tool would cost tokens
    rule = parse_requires(ex.get("tool_schemas_text", "")).get(cand["name"])
    if rule:
        lines = ["TOOLS:"] + [f"  {s}" + (f"  # Requires: {rule}" if s.startswith(cand["name"] + "(") else "") for s in tool_strs]
    else:
        lines = ["TOOLS: " + ("; ".join(tool_strs) if tool_strs else "(none listed)")]

    # 2. calls that fall before the window (one line, most recent last)
    before = [t for t in turns[:start] if t.get("role") == "assistant_tool_call"]
    if before and earlier:
        lines.append("EARLIER_CALLS: " + "; ".join(_call(t["name"], t["arguments"], arg_chars) for t in before[-earlier:]))

    # 3. the window
    lines.append("HISTORY:")
    for t in window:
        role = t.get("role")
        if role == "user":
            lines.append("USER: " + _clip(t.get("content", ""), msg_chars))
        elif role == "assistant":
            lines.append("AGENT: " + _clip(t.get("content", ""), msg_chars))
        elif role == "assistant_tool_call":
            lines.append("AGENT_CALL: " + _call(t["name"], t["arguments"], arg_chars))
        elif role == "tool":
            lines.append("TOOL_RESULT: " + _clip(t.get("content", ""), tool_chars))
        else:
            lines.append(f"{str(role).upper()}: " + _clip(t.get("content", ""), msg_chars))

    # 4. candidate, always last
    lines.append("CANDIDATE: " + _call(cand["name"], cand["arguments"], CAND_ARG_CHARS))
    return "\n".join(lines)


def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def format_input(example, tok=True):
    # Returns the model input string. With tok=True the text is tightened until it fits TOKEN_BUDGET.
    for lvl in LEVELS:
        text = _format(example, **lvl)
        if not tok or n_tokens(text) <= TOKEN_BUDGET:
            return text
    return text


# --- token length distribution over 200 random training examples ---
sample = random.Random(SEED).sample(rows, 200)
natural = np.array([n_tokens(_format(r, **LEVELS[0])) for r in sample])
final = np.array([n_tokens(format_input(r)) for r in sample])
pct = lambda a: {f"p{p}": int(np.percentile(a, p)) for p in (50, 90, 99)} | {"max": int(a.max())}
print("tokens, spec settings only :", pct(natural), f"| over {TOKEN_BUDGET}: {(natural > TOKEN_BUDGET).sum()}/200")
print("tokens, after budget guard :", pct(final), f"| over {TOKEN_BUDGET}: {(final > TOKEN_BUDGET).sum()}/200")

# --- does the evidence survive compression? Class 4 needs the earlier identical call to be visible ---
def dup_visible(r, earlier):
    # is the candidate call (formatted exactly as in the history) present before the CANDIDATE line?
    lvl = {**LEVELS[0], "earlier": earlier}
    cs = _call(r["candidate_tool_call"]["name"], r["candidate_tool_call"]["arguments"], lvl["arg_chars"])
    return cs in _format(r, **lvl).split("CANDIDATE:")[0]

c4 = [r for r in rows if r["label"] == "incorrect" and r["error_class"] == 4]
print(f"class 4 negatives with the earlier duplicate visible: last-10 window only {sum(dup_visible(r, 0) for r in c4)}/{len(c4)}"
      f"  ->  with EARLIER_CALLS {sum(dup_visible(r, LEVELS[0]['earlier']) for r in c4)}/{len(c4)}")

ex = next(r for r in c4 if len(r["turns"]) > 14)
print("\n--- one formatted example (class 4 negative) ---")
print(format_input(ex)[:3000])


# ---------------------------------------------------------------- split (Phase 5 logic and seed)
from sklearn.model_selection import StratifiedGroupKFold

groups = [f"{r['domain']}::{r['task_id']}" for r in rows]
y = [LABEL2ID[r["label"]] for r in rows]
# --- frozen split (begin) ---
# The grouped fold StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42) returns depends on the
# scikit-learn version: 6,816 train / 764 val from 1.8 on (the split D-012 documents), 6,892 / 688 up to 1.7.
# So the validation groups are frozen here (computed with scikit-learn 1.9.1 by notebooks/freeze_split.py) and
# every environment trains on the same split. Every row of a (domain, task_id) group goes to the same side.
VAL_GROUPS = {
    "bfcl::multi_turn_base_108",
    "bfcl::multi_turn_base_145",
    "bfcl::multi_turn_base_154",
    "bfcl::multi_turn_base_155",
    "bfcl::multi_turn_base_171",
    "bfcl::multi_turn_base_184",
    "bfcl::multi_turn_base_186",
    "bfcl::multi_turn_base_38",
    "bfcl::multi_turn_base_68",
    "bfcl::multi_turn_base_69",
    "bfcl::multi_turn_base_76",
    "bfcl::multi_turn_base_85",
    "bfcl::multi_turn_base_95",
    "bfcl::multi_turn_long_context_105",
    "bfcl::multi_turn_long_context_110",
    "bfcl::multi_turn_long_context_154",
    "bfcl::multi_turn_long_context_156",
    "bfcl::multi_turn_long_context_184",
    "bfcl::multi_turn_miss_func_154",
    "bfcl::multi_turn_miss_func_32",
    "bfcl::multi_turn_miss_func_44",
    "bfcl::multi_turn_miss_param_0",
    "bfcl::multi_turn_miss_param_1",
    "bfcl::multi_turn_miss_param_100",
    "bfcl::multi_turn_miss_param_109",
    "bfcl::multi_turn_miss_param_118",
    "bfcl::multi_turn_miss_param_131",
    "bfcl::multi_turn_miss_param_133",
    "bfcl::multi_turn_miss_param_136",
    "bfcl::multi_turn_miss_param_15",
    "bfcl::multi_turn_miss_param_152",
    "bfcl::multi_turn_miss_param_159",
    "bfcl::multi_turn_miss_param_169",
    "bfcl::multi_turn_miss_param_17",
    "bfcl::multi_turn_miss_param_176",
    "bfcl::multi_turn_miss_param_179",
    "bfcl::multi_turn_miss_param_18",
    "bfcl::multi_turn_miss_param_192",
    "bfcl::multi_turn_miss_param_28",
    "bfcl::multi_turn_miss_param_3",
    "bfcl::multi_turn_miss_param_36",
    "bfcl::multi_turn_miss_param_38",
    "bfcl::multi_turn_miss_param_39",
    "bfcl::multi_turn_miss_param_51",
    "bfcl::multi_turn_miss_param_73",
    "bfcl::multi_turn_miss_param_76",
    "bfcl::multi_turn_miss_param_96",
    "retail::11",
    "retail::21",
    "retail::27",
    "retail::39",
    "retail::64",
    "retail::68",
    "retail::81",
    "telecom::[mms_issue]bad_wifi_calling|break_apn_mms_setting|break_app_both_permissions|data_mode_off|data_usage_exceeded|user_abroad_roaming_disabled_off[PERSONA:None]",
    "telecom::[mobile_data_issue]bad_network_preference|bad_vpn|user_abroad_roaming_enabled_off[PERSONA:Easy]",
    "telecom::[mobile_data_issue]data_saver_mode_on|data_usage_exceeded[PERSONA:Easy]",
    "telecom::[mobile_data_issue]data_usage_exceeded|user_abroad_roaming_enabled_off[PERSONA:Easy]",
}
if DATASET == "final":  # the final dataset carries its split in every row (D-022)
    train_idx = np.array([i for i, r in enumerate(rows) if r["split"] == "train"])
    val_idx = np.array([i for i, r in enumerate(rows) if r["split"] == "val"])
else:
    assert VAL_GROUPS <= set(groups), "frozen validation groups that are not in the data: the data changed, rerun freeze_split.py"
    val_idx = np.array([i for i, g in enumerate(groups) if g in VAL_GROUPS])
    train_idx = np.array([i for i, g in enumerate(groups) if g not in VAL_GROUPS])
# --- frozen split (end) ---
train_rows = [rows[i] for i in train_idx]
val_rows = [rows[i] for i in val_idx]

# --- integrity checks ---
assert not ({groups[i] for i in train_idx} & {groups[i] for i in val_idx}), "a group straddles train and val"
split_of = {rows[i]["trajectory_id"]: "train" for i in train_idx} | {rows[i]["trajectory_id"]: "val" for i in val_idx}
pairs = [r for r in rows if r.get("paired_negative_id")]
missing = [r["trajectory_id"] for r in pairs if r["paired_negative_id"] not in split_of]
assert not missing, f"{len(missing)} controls point at a negative that is not in the dataset"
assert all(split_of[r["trajectory_id"]] == split_of[r["paired_negative_id"]] for r in pairs), "a control pair was split"
by_tier = Counter(r["label_tier"] for r in pairs)
if DATASET == "legacy":
    assert set(by_tier) == {"synthetic_class2_control", "synthetic_class4_control", "synthetic_class6_control"}, by_tier
for cls in (2, 4, 6):
    ps = [r for r in pairs if r["label_tier"] == f"synthetic_class{cls}_control"]
    n_tr = sum(split_of[r["trajectory_id"]] == "train" for r in ps)
    print(f"Class {cls} pairs confirmed in the same split (via paired_negative_id): {len(ps)}  (train {n_tr}, val {len(ps) - n_tr})")
n_neg = {cls: sum(r["label"] == "incorrect" and r["error_class"] == cls for r in rows) for cls in (2, 4, 6)}
print("negatives without a control (no twin, so nothing to pair):", {f"class {c}": n_neg[c] - sum(r["label_tier"] == f"synthetic_class{c}_control" for r in pairs) for c in (2, 4, 6)})

for name, part in (("train", train_rows), ("val", val_rows)):
    n_inc = sum(r["label"] == "incorrect" for r in part)
    print(f"\n{name}: {len(part)} rows ({len(part) / len(rows):.1%})  incorrect={n_inc} ({n_inc / len(part):.1%})  correct={len(part) - n_inc}")
    print("  domains:", dict(Counter(r["domain"] for r in part)))
    print("  incorrect by class:", dict(sorted(Counter(r["error_class"] for r in part if r["label"] == "incorrect").items())))
print(f"\ngroups: train={len(set(groups[i] for i in train_idx))}  val={len(set(groups[i] for i in val_idx))}")

if DATASET == "legacy":
    assert (len(rows), len(train_rows), len(val_rows)) == (7580, 6816, 764), (len(rows), len(train_rows), len(val_rows))  # the frozen split, D-012
print(f"split sizes: train {len(train_rows)} / val {len(val_rows)}" + (" (documented 6,816 / 764)" if DATASET == "legacy" else " (taken from the split field of the rows)"))

if SMOKE:  # tiny subset so the whole notebook runs on a laptop CPU in a minute
    train_rows, val_rows = train_rows[:192], val_rows[:64]
    print("SMOKE: using", len(train_rows), "train /", len(val_rows), "val rows")

## Cell 4: tokenize, LoRA fine-tune, save the best adapter

`Qwen/Qwen2.5-1.5B` as a sequence classifier (last-token pooling: the `CANDIDATE:` line is the last thing in every input), 4-bit NF4, LoRA r=16 / alpha=32 / dropout 0.05 on `q_proj` and `v_proj`; the classification head (`score`) is trained too (PEFT saves it with the adapter). lr 2e-4, 3 epochs, effective batch 16 (micro-batch x accumulation), 10% warmup, weight decay 0.01, fp16. Speed (D-020): length-bucketed training batches, length-sorted validation, gradient checkpointing ON by default with micro-batch 2 (`GRADIENT_CHECKPOINTING`, `MICRO_BATCH` in the first cell; the retry ladder only goes down, 2 then 1, and a failed attempt is cleaned up before the next one, D-029), and `BENCHMARK_STEPS` for a timing run that stops without saving. The best epoch by validation F1 (positive class = `incorrect`) is saved as a LoRA adapter plus the tokenizer.

In [ ]:
LR, EPOCHS, WARMUP_RATIO, EFFECTIVE_BATCH = 2e-4, (1 if SMOKE else 3), 0.1, 16
TRAINER_DIR = CKPT_DIR / "trainer"

# --- length bucketing (D-020) ---
import random as _random


def padded_tokens(lengths, batches, multiple=8):
    """Tokens the model processes for these batches: every row is padded to the longest row of its batch (rounded up to `multiple`)."""
    total = 0
    for b in batches:
        longest = max(lengths[i] for i in b)
        total += len(b) * (-(-longest // multiple) * multiple)
    return total


class LengthBucketSampler:
    """transformers 5.x removed `group_by_length`, so this does it. Each epoch: a fresh random permutation is cut into chunks of
    `batch_size * mega` rows, each chunk is sorted by length and cut into batches, and then ALL batches are shuffled, so a batch holds
    rows of similar length and the order between batches is random. The batch with the longest row goes first: an out-of-memory error
    shows at step 1, and every epoch has the same worst-case batch shape. Yields row indices; every run of `batch_size` consecutive
    indices is one batch (a smaller leftover batch is last)."""

    def __init__(self, lengths, batch_size, seed=42, mega=50):
        self.lengths, self.batch_size, self.seed, self.mega, self.epoch = list(lengths), batch_size, seed, mega, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def batches(self, epoch=None):
        epoch = self.epoch if epoch is None else epoch
        rng = _random.Random(self.seed * 1000003 + epoch)
        idx = list(range(len(self.lengths)))
        rng.shuffle(idx)
        chunk, bs, out = self.batch_size * self.mega, self.batch_size, []
        for s in range(0, len(idx), chunk):
            part = sorted(idx[s:s + chunk], key=lambda i: -self.lengths[i])
            out += [part[j:j + bs] for j in range(0, len(part), bs)]
        full, rest = [b for b in out if len(b) == bs], [b for b in out if len(b) < bs]
        rng.shuffle(full)
        if full:
            k = max(range(len(full)), key=lambda j: max(self.lengths[i] for i in full[j]))
            full.insert(0, full.pop(k))
        rest = [i for b in rest for i in b]
        return full + ([rest] if rest else [])  # the leftover rows (fewer than one batch) form the last batch

    def __iter__(self):
        for b in self.batches(self.epoch):
            yield from b
        self.epoch += 1

    def __len__(self):
        return len(self.lengths)


class LengthSortedSampler:
    """Evaluation order: shortest rows first, so every eval batch has rows of about the same length (the metrics do not depend on the order)."""

    def __init__(self, lengths):
        self.order = sorted(range(len(lengths)), key=lambda i: lengths[i])

    def __iter__(self):
        return iter(self.order)

    def __len__(self):
        return len(self.order)


import statistics
import time
from transformers import TrainerCallback


def row_lengths(ds):
    """Token count of every row of a tokenized dataset (a plain dataset object with .input_ids, or a Hugging Face Dataset)."""
    ids = ds["input_ids"] if hasattr(ds, "column_names") else ds.input_ids
    return [len(x) for x in ids]


def padding_report(lengths, batch):
    """Padding waste of the training set at this loader batch size: random order (what the Trainer did before) against length-bucketed."""
    real, rnd = sum(lengths), 0
    for s in range(3):
        o = list(range(len(lengths)))
        _random.Random(s).shuffle(o)
        rnd += padded_tokens(lengths, [o[i:i + batch] for i in range(0, len(o), batch)]) / 3
    buck = padded_tokens(lengths, LengthBucketSampler(lengths, batch).batches(0))
    print(f"padding waste at loader batch {batch}: random order {1 - real / rnd:.1%} ({rnd:,.0f} tokens processed for {real:,} real) -> "
          f"length-bucketed {1 - real / buck:.1%} ({buck:,.0f}); an epoch now processes {buck / rnd:.0%} of the tokens it did before")


class BenchmarkCallback(TrainerCallback):
    """BENCHMARK_STEPS mode: times every optimizer step (data loading included) and stops training after `n` of them."""

    def __init__(self, n):
        self.n, self.times, self.t0 = n, [], None

    def on_step_begin(self, args, state, control, **kw):
        self.t0 = time.perf_counter()

    def on_step_end(self, args, state, control, **kw):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.times.append(time.perf_counter() - self.t0)
        if len(self.times) >= self.n:
            control.should_training_stop = True


class LengthBucketTrainer(Trainer):
    """Trainer with length-bucketed training batches and length-sorted evaluation batches (D-020). It overrides two hooks that exist in
    transformers 4.x and 5.x and does not use `group_by_length` (its name and place changed between versions)."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.bench = None
        if BENCHMARK_STEPS:
            self.bench = BenchmarkCallback(BENCHMARK_STEPS)
            self.add_callback(self.bench)

    def _get_train_sampler(self, *args, **kwargs):
        ds = args[0] if args and args[0] is not None else self.train_dataset
        return LengthBucketSampler(row_lengths(ds), self._train_batch_size, seed=self.args.seed)

    def _get_eval_sampler(self, eval_dataset, *args, **kwargs):
        return LengthSortedSampler(row_lengths(eval_dataset))


def best_eval_metrics(trainer):
    """Validation metrics of the epoch that load_best_model_at_end restored, read from the per-epoch evaluations (no extra pass)."""
    step = getattr(trainer.state, "best_global_step", None)
    for h in trainer.state.log_history:
        if "eval_f1" in h and step and h.get("step") == step:
            return {k: v for k, v in h.items() if k.startswith("eval_")}
    print("best epoch not found in the log history: evaluating once more")
    return {k: v for k, v in trainer.evaluate().items() if k.startswith("eval_")}


def benchmark_report(trainer, n_gpu, batch_size, accum, ckpt):
    """BENCHMARK_STEPS mode: seconds per step, the projected time of a full run, then stop without saving anything."""
    t = trainer.bench.times
    skip = min(2, len(t) - 1)  # step 1 holds the longest batch and the CUDA warm-up
    use = t[skip:]
    mean, se = statistics.fmean(use), (statistics.pstdev(use) / len(use) ** 0.5 if len(use) > 1 else 0.0)
    t0 = time.perf_counter()
    trainer.evaluate()
    eval_s = time.perf_counter() - t0
    steps = math.ceil(len(train_ds) / EFFECTIVE_BATCH) * EPOCHS
    saves = EPOCHS * 90  # writing a checkpoint with optimizer state takes about a minute or two
    total = mean * steps + eval_s * EPOCHS + saves
    peak = f"{torch.cuda.max_memory_allocated() / 2**30:.1f} GiB peak GPU memory (one device)" if torch.cuda.is_available() else "no GPU"
    print("\n" + "=" * 100)
    print(f"BENCHMARK: {len(t)} optimizer steps | {n_gpu} GPU(s) x micro-batch {batch_size} x accumulation {accum} = {EFFECTIVE_BATCH} | gradient checkpointing {'on' if ckpt else 'off'} | {peak}")
    print(f"  step 1 (longest batch, warm-up): {t[0]:.1f}s | steps {skip + 1}-{len(t)}: mean {mean:.2f}s +- {se:.2f}s per step, min {min(use):.2f}s, max {max(use):.2f}s")
    print(f"  {EFFECTIVE_BATCH / mean:.2f} rows/s | validation pass {eval_s:.0f}s (length-sorted)")
    print(f"  full run = {steps} steps ({len(train_ds)} rows x {EPOCHS} epochs): {steps} x {mean:.2f}s + {EPOCHS} x {eval_s:.0f}s validation + about {saves // 60} min of saves "
          f"= {total / 3600:.2f} h ({int(total):,} s)")
    budget = globals().get("TIME_BUDGET_HOURS")
    if budget:
        print(f"  that is {'inside' if total / 3600 < budget else 'OUTSIDE'} the {budget} h time budget of this notebook")
    print("=" * 100)
    raise SystemExit("BENCHMARK_STEPS mode: stopped after the benchmark, nothing was saved. Set BENCHMARK_STEPS = None for the real run.")

# Retry ladder (D-029): starts at MICRO_BATCH with the GRADIENT_CHECKPOINTING setting of the first cell and only goes DOWN (halving the micro-batch, never switching
# checkpointing, never a larger batch); every entry gives an effective batch of 16 (accumulation = 16 / micro-batch). The longest batch comes first in every epoch
# (LengthBucketSampler), so a memory problem shows at step 1, not after hours.
_CKPT = True if GRADIENT_CHECKPOINTING is None else bool(GRADIENT_CHECKPOINTING)
_START = 2 if MICRO_BATCH is None else int(MICRO_BATCH)
assert _START in (1, 2, 4, 8, 16), f"MICRO_BATCH must be 1, 2, 4, 8 or 16 (it must divide the effective batch of 16), not {MICRO_BATCH}"
LADDER = [(b, _CKPT) for b in (16, 8, 4, 2, 1) if b <= _START]
EVAL_BATCH = 2   # validation batch (forward only, longest rows 4,096 tokens): 2 keeps the working set to about 1 GiB
LEAK_LIMIT_GIB = 0.5   # GPU memory still allocated after the cleanup of a failed attempt that makes the notebook stop instead of retrying (a real leak)
ATTEMPTS = []          # one entry per training attempt: micro-batch, checkpointing, outcome, peak memory (goes into train_info.json)
# --- end length bucketing ---
# --- memory-safe training attempts (D-029) ---
def host_mem():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f"host RAM used {(vm.total - vm.available) / 2 ** 30:.1f} of {vm.total / 2 ** 30:.1f} GiB, this process {psutil.Process().memory_info().rss / 2 ** 30:.1f} GiB"
    except Exception:
        return "host RAM unknown"


def gpu_mem(tag):
    """One line with the GPU memory (what PyTorch has allocated / reserved / peaked at, what the driver says is free) and the host RAM."""
    g = 2 ** 30
    if not torch.cuda.is_available():
        print(f"[memory | {tag}] no GPU | {host_mem()}", flush=True)
        return
    free, total = torch.cuda.mem_get_info()
    print(f"[GPU memory | {tag}] allocated {torch.cuda.memory_allocated() / g:.2f} GiB | reserved {torch.cuda.memory_reserved() / g:.2f} GiB | "
          f"peak allocated {torch.cuda.max_memory_allocated() / g:.2f} GiB | free (driver) {free / g:.2f} of {total / g:.2f} GiB | {host_mem()}", flush=True)


try:
    import faulthandler
    faulthandler.enable()  # a native crash (segfault, abort) prints the Python stack instead of ending the cell silently
except Exception:
    pass


def release_gpu():
    """Free what is unreachable now. Only effective when nothing refers to the failed attempt any more (train_attempt guarantees that by returning before this runs)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or (isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower())


class MemoryLog(TrainerCallback):
    """Prints the GPU memory after the first optimizer step (the longest batch, so the worst case of the whole run) and after step 10."""

    def on_step_end(self, args, state, control, **kw):
        if state.global_step in (1, 10):
            gpu_mem(f"after optimizer step {state.global_step}")


def _dispose(trainer):
    """Drop the references that keep a failed attempt's model, optimizer and gradients alive (the Trainer, its accelerator and the callbacks form reference cycles)."""
    if trainer is None:
        return
    try:
        trainer.accelerator.free_memory()  # accelerate's own release of the prepared models / optimizers
    except Exception:
        pass
    for name in ("model", "model_wrapped", "optimizer", "lr_scheduler", "train_dataset", "eval_dataset"):
        try:
            setattr(trainer, name, None)
        except Exception:
            pass


def train_attempt(make_trainer, banner, resume, pad_batch):
    """Build the trainer and train, all inside this function. Returns (trainer, None) on success and (None, message) after an out-of-memory error.
    The cleanup is NOT inside the `except` block: while the handler runs, the exception still holds its traceback and with it the frames of the failed step (activations,
    gradients, the model), which would keep all of that on the GPU. Leaving the block releases the traceback; only then are the trainer's references dropped."""
    trainer, why = None, None
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        trainer = make_trainer()
        print(banner, flush=True)
        padding_report(row_lengths(train_ds), pad_batch)
        trainer.train(resume_from_checkpoint=resume)
        return trainer, None
    except Exception as exc:
        if not is_oom(exc):
            raise
        why = f"{type(exc).__name__}: {str(exc).splitlines()[0][:230] if str(exc) else ''}"
    _dispose(trainer)
    trainer = None
    return None, why


def run_ladder(make_trainer_for, resume, n_gpu):
    """Try the entries of LADDER in order. `make_trainer_for(batch_size, ckpt, accum)` returns a fresh Trainer. Returns (trainer, batch_size, ckpt, accum)."""
    progress_callbacks = [MemoryLog()]
    base = torch.cuda.memory_allocated() / 2 ** 30 if torch.cuda.is_available() else 0.0
    if base > LEAK_LIMIT_GIB:
        print(f"WARNING: {base:.2f} GiB of GPU memory are already allocated before the first attempt (objects of an earlier run of a cell in this session). "
              "For a clean start use Run > Restart session; the leak check below counts only what a failed attempt adds to this.", flush=True)
    for k, (batch_size, ckpt) in enumerate(LADDER, 1):
        assert EFFECTIVE_BATCH % (batch_size * max(n_gpu, 1)) == 0, f"micro-batch {batch_size} x {n_gpu} GPU does not divide the effective batch {EFFECTIVE_BATCH}"
        accum = EFFECTIVE_BATCH // (batch_size * max(n_gpu, 1))
        gpu_mem(f"before attempt {k}/{len(LADDER)}: micro-batch {batch_size}, checkpointing {'on' if ckpt else 'off'}")
        banner = (f"\ntraining with {n_gpu} GPU x per-device batch {batch_size} x grad_accum {accum} = {EFFECTIVE_BATCH}, gradient checkpointing {'on' if ckpt else 'off'} "
                  f"(attempt {k}/{len(LADDER)})")
        trainer, why = train_attempt(lambda: make_trainer_for(batch_size, ckpt, accum, progress_callbacks), banner, resume, batch_size * max(n_gpu, 1))
        if trainer is not None:
            peak = torch.cuda.max_memory_allocated() / 2 ** 30 if torch.cuda.is_available() else None
            ATTEMPTS.append({"micro_batch": batch_size, "gradient_checkpointing": ckpt, "outcome": "ok", "peak_gpu_gib": peak})
            return trainer, batch_size, ckpt, accum
        peak = torch.cuda.max_memory_allocated() / 2 ** 30 if torch.cuda.is_available() else None
        ATTEMPTS.append({"micro_batch": batch_size, "gradient_checkpointing": ckpt, "outcome": "out of memory", "peak_gpu_gib": peak, "message": why})
        print(f"CUDA out of memory with micro-batch {batch_size} (checkpointing {'on' if ckpt else 'off'}); peak {peak if peak is None else format(peak, '.2f')} GiB | {why}", flush=True)
        gpu_mem("failed attempt, before cleanup")
        release_gpu()
        gpu_mem("after cleanup")
        if torch.cuda.is_available() and torch.cuda.memory_allocated() / 2 ** 30 - base > LEAK_LIMIT_GIB:
            nxt = LADDER[k][0] if k < len(LADDER) else None
            raise RuntimeError(
                f"{torch.cuda.memory_allocated() / 2 ** 30 - base:.2f} GiB of GPU memory are still allocated after the cleanup of the failed attempt (limit {LEAK_LIMIT_GIB} GiB): something still "
                "refers to it, and a retry in this process would start with less memory than the one that failed. Restart the session (Kaggle: Run > Restart session; "
                "Colab: Runtime > Restart session) and set "
                + (f"MICRO_BATCH = {nxt}" if nxt else "QWEN_MAX_LEN = 2048 (Kaggle) / a shorter input (Colab)") + " in the first cell.")
    raise RuntimeError(
        f"out of memory at every micro-batch of the ladder {[b for b, _ in LADDER]} with gradient checkpointing {'on' if LADDER[0][1] else 'off'}. See the attempts above: "
        "the peak memory of the smallest micro-batch tells how far it is from fitting. Lower QWEN_MAX_LEN (Kaggle) / the input length (Colab), or use MODEL_PATH_OVERRIDE = 'Qwen/Qwen2.5-0.5B'.")
# --- end memory-safe training attempts ---

assert torch.cuda.is_available() or SMOKE, "Select a GPU runtime (Runtime > Change runtime type > T4 GPU): 4-bit training needs CUDA"


def encode(part):
    texts = [format_input(r) for r in part]
    n_cut = sum(len(ids) > MAX_LEN for ids in tokenizer(texts)["input_ids"])
    assert n_cut == 0, f"{n_cut} inputs are longer than MAX_LEN and truncation would cut off their CANDIDATE line"
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN)  # padding is dynamic, per batch, by the collator
    return Dataset.from_dict({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                              "labels": [LABEL2ID[r["label"]] for r in part]})


def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision": prec, "recall": rec, "f1": f1,
            "auroc": roc_auc_score(p.label_ids, probs)}


def load_base_model():
    # The 4-bit backbone plus a freshly initialised 2-way `score` head. For evaluation the saved adapter
    # (which includes the trained head) is loaded on top of this.
    if SMOKE:  # randomly initialised toy model: exercises the code path only (no 4-bit without a CUDA GPU)
        from transformers import Qwen2Config, Qwen2ForSequenceClassification
        set_seed(SEED)
        return Qwen2ForSequenceClassification(Qwen2Config(
            vocab_size=len(tokenizer), hidden_size=64, intermediate_size=128, num_hidden_layers=2, num_attention_heads=2,
            num_key_value_heads=2, num_labels=2, pad_token_id=tokenizer.pad_token_id, id2label=ID2LABEL, label2id=LABEL2ID))
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True, llm_int8_skip_modules=["score"])
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
        pad_token_id=tokenizer.pad_token_id,  # the config has none, and the head needs it to find the last real token
        quantization_config=bnb, device_map={"": 0}, trust_remote_code=True, attn_implementation="sdpa",  # sdpa explicitly: an eager-attention fallback needs ~2 GiB more per 4,096-token row
        torch_dtype=torch.float16)  # explicit: the default would load Qwen's bf16 weights, which a T4 does not support
    model.config.use_cache = False
    return model


def make_peft_model(ckpt=True):
    set_seed(SEED)
    base = load_base_model()
    if not SMOKE:  # casts the non-quantized weights to fp32; gradient checkpointing only when `ckpt` (it is the memory fallback, D-020)
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=ckpt, gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05,
                                            bias="none", task_type=TaskType.SEQ_CLS))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()  # fp16 mixed precision cannot unscale fp16 gradients, so trainable weights stay fp32
    assert any("score" in n for n, p in model.named_parameters() if p.requires_grad), "the classification head is not trainable"
    model.print_trainable_parameters()
    return model


train_ds, val_ds = encode(train_rows), encode(val_rows)
print("train/val examples:", len(train_ds), len(val_ds), "| longest tokenized input:", max(len(x) for x in train_ds["input_ids"]), "tokens")


def build_trainer(batch_size, accum, ckpt):
    steps = math.ceil(len(train_ds) / (batch_size * accum)) * EPOCHS
    args = TrainingArguments(
        output_dir=str(TRAINER_DIR), num_train_epochs=EPOCHS, learning_rate=LR, optim="adamw_torch",
        lr_scheduler_type="linear", warmup_steps=int(WARMUP_RATIO * steps),  # 10% warmup; current transformers has no warmup_ratio
        weight_decay=0.01, per_device_train_batch_size=batch_size, per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=accum, eval_strategy=("no" if BENCHMARK_STEPS else "epoch"), save_strategy=("no" if BENCHMARK_STEPS else "epoch"), save_total_limit=2,
        load_best_model_at_end=not BENCHMARK_STEPS, metric_for_best_model="f1", greater_is_better=True,
        logging_steps=(5 if SMOKE else 50), fp16=torch.cuda.is_available(), gradient_checkpointing=(ckpt and not SMOKE),
        gradient_checkpointing_kwargs={"use_reentrant": False}, report_to="none", seed=SEED)
    return LengthBucketTrainer(model=make_peft_model(ckpt), args=args, train_dataset=train_ds, eval_dataset=val_ds,
                   processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
                   compute_metrics=compute_metrics)


def print_history(log_history):
    train_loss, evals = {}, []
    for h in log_history:
        if "loss" in h and "eval_loss" not in h:
            train_loss.setdefault(math.ceil(h["epoch"] - 1e-9), []).append(h["loss"])
        if "eval_f1" in h:
            evals.append(h)
    print("epoch  train loss  val loss  accuracy  precision  recall     F1   AUROC")
    for h in evals:
        e = round(h["epoch"])
        tl = np.mean(train_loss[e]) if train_loss.get(e) else float("nan")
        print(f"{e:5d}  {tl:10.4f}  {h['eval_loss']:8.4f}  {h['eval_accuracy']:8.3f}  {h['eval_precision']:9.3f}  {h['eval_recall']:6.3f}  {h['eval_f1']:5.3f}  {h['eval_auroc']:6.3f}")
    if evals:
        b = max(evals, key=lambda h: h["eval_f1"])
        print(f"best epoch by validation F1: {round(b['epoch'])} (F1 {b['eval_f1']:.3f}, AUROC {b['eval_auroc']:.3f})")


if (BEST_DIR / "adapter_config.json").exists():
    print(f"Found a trained adapter at {BEST_DIR}; skipping training.")
    if (BEST_DIR / "train_info.json").exists():
        print_history(json.load(open(BEST_DIR / "train_info.json"))["log_history"])
else:
    resume = get_last_checkpoint(str(TRAINER_DIR)) if TRAINER_DIR.exists() else None
    print("resuming from", resume) if resume else print("starting fresh")
    def make_trainer_for(batch_size, ckpt, accum, callbacks):
        t = build_trainer(batch_size, accum, ckpt)
        for cb in callbacks:
            t.add_callback(cb)
        return t

    trainer, batch_size, ckpt, accum = run_ladder(make_trainer_for, resume, 1)

    if BENCHMARK_STEPS:
        try:
            benchmark_report(trainer, 1, batch_size, accum, ckpt)  # prints, then stops: nothing is saved
        finally:  # free the GPU: a later run of this cell in the same session must not start next to this attempt's model and optimizer
            trainer = None
            release_gpu()
    trainer.save_model(str(BEST_DIR))  # the best epoch by val F1 (load_best_model_at_end): LoRA adapter + trained head
    tokenizer.save_pretrained(str(BEST_DIR))
    history = list(trainer.state.log_history)  # per-epoch history, taken before evaluate() below logs one more eval entry
    best = best_eval_metrics(trainer)
    (BEST_DIR / "train_info.json").write_text(json.dumps({
        "model": MODEL_NAME, "val_metrics": best, "lr": LR, "epochs": EPOCHS, "effective_batch": EFFECTIVE_BATCH,
        "micro_batch": batch_size, "grad_accum": accum, "gradient_checkpointing": ckpt, "attempts": ATTEMPTS, "peak_gpu_gib": (round(torch.cuda.max_memory_allocated() / 2 ** 30, 2) if torch.cuda.is_available() else None), "max_len": MAX_LEN, "token_budget": TOKEN_BUDGET, "seed": SEED,
        "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "targets": ["q_proj", "v_proj"]},
        "n_train": len(train_ds), "n_val": len(val_ds), "log_history": history}, indent=1, default=str))
    print()
    print_history(history)
    print("\nsaved adapter + tokenizer to", BEST_DIR)

## Cell 5: evaluate on `test_real.jsonl`

Identical to Phase 5's evaluation (positive class = `incorrect`; "false-flag rate" = share of the *correct* rows flagged as incorrect), except that the model is the 4-bit backbone plus the saved LoRA adapter. Remember (D-010): 25 incorrect rows, so per-class and per-domain numbers are very noisy; the correct half is "suspicious-looking but fine" calls; and the 8 real Class 7 rows were never trained on.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

eval_tok = AutoTokenizer.from_pretrained(str(BEST_DIR))
eval_tok.padding_side = "right"
model = PeftModel.from_pretrained(load_base_model(), str(BEST_DIR))  # 4-bit backbone + the saved adapter and head
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"  # the quantized model is already on the GPU

TEST_NAME = "test_real" if DATASET == "legacy" else "the final test set"
PRED_FILE = "test_real_predictions.jsonl" if DATASET == "legacy" else "test_predictions.jsonl"
test_rows = load_jsonl(DATA / "test_real.jsonl") if DATASET == "legacy" else FINAL["test"]  # same file as Cell 2
y_true = np.array([LABEL2ID[r["label"]] for r in test_rows])
test_texts = [format_input(r) for r in test_rows]


@torch.no_grad()
def predict_probs(texts, batch_size=4):  # 4 rows of up to 4,096 tokens, forward only
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))  # similar lengths together
    out = np.zeros(len(texts))
    for s in range(0, len(order), batch_size):
        idx = order[s : s + batch_size]
        enc = eval_tok([texts[i] for i in idx], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**enc).logits
        out[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return out


probs = predict_probs(test_texts)


def metrics_at(t, mask=None):
    m = np.ones(len(y_true), bool) if mask is None else mask
    yt, pr = y_true[m], (probs[m] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
    p, r, f, _ = precision_recall_fscore_support(yt, pr, average="binary", pos_label=1, zero_division=0)
    return dict(acc=(tp + tn) / len(yt), precision=p, recall=r, f1=f, false_flag=fp / max(fp + tn, 1), tn=tn, fp=fp, fn=fn, tp=tp)


THRESH = 0.5
o = metrics_at(THRESH)
print(f"=== overall on {TEST_NAME} (n={len(y_true)}, {int(y_true.sum())} incorrect), threshold {THRESH} ===")
print(f"accuracy {o['acc']:.3f}  precision {o['precision']:.3f}  recall {o['recall']:.3f}  F1 {o['f1']:.3f}  AUROC {roc_auc_score(y_true, probs):.3f}")
print(f"(always predicting 'correct' would score accuracy {1 - y_true.mean():.3f})")
PHASE5_AUROC = 0.652  # ModernBERT-base baseline, as given for this phase
auc = roc_auc_score(y_true, probs)
print(f"AUROC vs the Phase 5 baseline of {PHASE5_AUROC}: {auc:.3f} ({auc - PHASE5_AUROC:+.3f}), {'beats' if auc > PHASE5_AUROC else 'does not beat'} it")

print("\n=== recall per error class (incorrect rows only) ===")
for c in range(1, 8):
    m = np.array([r["label"] == "incorrect" and r["error_class"] == c for r in test_rows])
    if m.sum() == 0:
        print(f"  class {c}: n=0 (no real examples)")
    else:
        print(f"  class {c}: recall {(probs[m] >= THRESH).mean():.2f}  (n={int(m.sum())})")

print("\n=== per domain ===")
for d in ("retail", "airline", "telecom"):
    m = np.array([r["domain"] == d for r in test_rows])
    dm = metrics_at(THRESH, m)
    print(f"  {d:8s} n={int(m.sum()):3d} (incorrect={int(y_true[m].sum()):2d})  accuracy {dm['acc']:.3f}  F1 {dm['f1']:.3f}")

print("\n=== confusion matrix (rows = truth, columns = prediction) ===")
print(f"                   pred correct   pred incorrect")
print(f"  true correct     {o['tn']:12d}   {o['fp']:14d}")
print(f"  true incorrect   {o['fn']:12d}   {o['tp']:14d}")

print("\n=== threshold sweep ===")
print("  thresh      F1   precision  recall  false-flag rate")
for t in (0.3, 0.5, 0.7):
    s = metrics_at(t)
    print(f"  {t:5.1f}   {s['f1']:7.3f}   {s['precision']:8.3f}  {s['recall']:6.3f}  {s['false_flag']:14.3f}")

if DATASET == "final":
    # ================= final-mode report (D-023): real test set, per class by origin, per domain, synthetic test, baselines =================
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_curve

    def _boot(y, p, n_boot=1000, seed=0):  # 95% percentile interval over row resamples (same method as before)
        rng = np.random.default_rng(seed)
        vals = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            if y[i].min() != y[i].max():
                vals.append(roc_auc_score(y[i], p[i]))
        return np.percentile(vals, [2.5, 97.5])

    def _recall_at_fpr(y, p, fpr_max):
        fpr, tpr, thr = roc_curve(y, p)
        k = np.flatnonzero(fpr <= fpr_max)[-1]
        return float(tpr[k]), float(thr[k])

    def _cell(flagged, n):  # fewer than 10 examples: do not show a rate
        return f"too few to read ({flagged}/{n})" if n < 10 else f"recall {flagged / n:.2f} ({flagged}/{n})"

    def _length(r):
        return float(np.log1p(len(json.dumps(r["turns"])) + len(json.dumps(r["candidate_tool_call"]))))

    def _src(r):
        s = str(r.get("source", ""))
        return "induced" if s.startswith("induced") else "critictool" if s == "critictool" else "bfcl" if r["domain"] == "bfcl" else "tau2"

    SRC_NAMES = ["tau2", "bfcl", "critictool", "induced"]
    base_tr = np.array([r["label"] == "incorrect" for r in FINAL["train"]]).astype(int)

    def _baselines(part, yy):
        out = {}
        for name, feat in (("length only", lambda r: [_length(r)]), ("source only", lambda r: [float(_src(r) == s) for s in SRC_NAMES])):
            clf = LogisticRegression(max_iter=1000).fit(np.array([feat(r) for r in FINAL["train"]]), base_tr)
            sc = clf.predict_proba(np.array([feat(r) for r in part]))[:, 1]
            out[name] = roc_auc_score(yy, sc) if yy.min() != yy.max() else float("nan")
        return out

    def _overall(title, part, yy, pr):
        print(f"\n=== {title}: {len(yy)} rows, {int(yy.sum())} incorrect ===")
        lo, hi = _boot(yy, pr)
        pred = pr >= THRESH
        tp, fp = int((pred & (yy == 1)).sum()), int((pred & (yy == 0)).sum())
        prec = tp / max(tp + fp, 1)
        rec = tp / max(int(yy.sum()), 1)
        print(f"AUROC {roc_auc_score(yy, pr):.3f}  95% CI {lo:.3f}-{hi:.3f}  (1,000 bootstrap resamples, seed 0)")
        print(f"at threshold {THRESH}: precision {prec:.3f}  recall {rec:.3f}  F1 {2 * prec * rec / max(prec + rec, 1e-9):.3f}  false-flag rate {fp / max(int((yy == 0).sum()), 1):.3f}")
        for f in (0.05, 0.10):
            r_, t_ = _recall_at_fpr(yy, pr, f)
            print(f"recall at {f:.0%} false-alarm rate: {r_:.3f} (threshold {t_:.3f})")
        b = _baselines(part, yy)
        print(f"baselines on the same rows (trained on the training split): length only AUROC {b['length only']:.3f}, source only AUROC {b['source only']:.3f}")

    _overall("REAL TEST SET (human-reviewed)", test_rows, y_true, probs)
    origins = sorted({r["origin"] for r in test_rows})
    print(f"\nrecall per error class by origin (threshold {THRESH}; a cell with fewer than 10 examples is not shown as a rate)")
    for c in range(1, 8):
        cells = []
        for o in origins:
            m = np.array([r["label"] == "incorrect" and r["error_class"] == c and r["origin"] == o for r in test_rows])
            cells.append(f"{o}: " + _cell(int((probs[m] >= THRESH).sum()), int(m.sum())))
        print(f"  class {c}: " + " | ".join(cells))
    # Class 7 = policy violation (docs/labeling_guide.md). MAIN NUMBER: overall Class 7 recall. The sub-reason split below is approximate: since 2026-11-15 the
    # reasons come from a heuristic pre-fill applied by rule (D-022), not from a human, so it is labeled that way.
    m7 = np.array([r["label"] == "incorrect" and r["error_class"] == 7 for r in test_rows])
    print(f"\nCLASS 7 (policy violation) RECALL, threshold {THRESH}: " + _cell(int((probs[m7] >= THRESH).sum()), int(m7.sum())) + "   <- the number to quote")
    print("  Class 7 recall by sub-reason (heuristic tags, approximate; a cell with fewer than 10 examples is not shown as a rate):")
    for src_name, src_title in (("human", "reason confirmed by a human"), ("heuristic", "heuristic tags, approximate"), ("generator", "reason set by the generator")):
        shown = False
        for reason in ("no_confirmation", "ineligible_state", "missing_step", "precondition_error", "parallel_write", "other", "unlabeled"):
            m = np.array([r["label"] == "incorrect" and r["error_class"] == 7 and r.get("class7_reason") == reason and r.get("class7_reason_source") == src_name for r in test_rows])
            if m.any():
                if not shown:
                    print(f"    {src_title}:")
                    shown = True
                print(f"      {reason:20s} " + _cell(int((probs[m] >= THRESH).sum()), int(m.sum())))
    print("\nper domain")
    for d in sorted({r["domain"] for r in test_rows}):
        m = np.array([r["domain"] == d for r in test_rows])
        ni, nc = int(y_true[m].sum()), int((1 - y_true[m]).sum())
        auc = f"AUROC {roc_auc_score(y_true[m], probs[m]):.3f}" if ni >= 10 and nc >= 10 else "AUROC too few to read"
        print(f"  {d:8s} n {int(m.sum()):4d} (incorrect {ni:3d}, correct {nc:3d})  {auc}  recall {(probs[m & (y_true == 1)] >= THRESH).mean() if ni else float('nan'):.2f}  false-flag {(probs[m & (y_true == 0)] >= THRESH).mean() if nc else float('nan'):.2f}")

    synth = FINAL["test_synth"]
    probs_s = predict_probs([format_input(r) for r in synth])
    ys = np.array([LABEL2ID[r["label"]] for r in synth])
    print("\n" + "-" * 78 + "\nSYNTHETIC TEST SET (held-out synthetic Classes 2 and 6 with their paired correct examples; NOT real errors)\n" + "-" * 78)
    _overall("SYNTHETIC held-out test", synth, ys, probs_s)
    for c in (2, 6):
        m = np.array([r["label"] == "incorrect" and r["error_class"] == c for r in synth])
        print(f"  synthetic class {c}: " + _cell(int((probs_s[m] >= THRESH).sum()), int(m.sum())))

    for name, part, pr in (("test_predictions.jsonl", test_rows, probs), ("test_synth_predictions.jsonl", synth, probs_s)):
        with open(CKPT_DIR / name, "w") as f:
            for r, p in zip(part, pr):
                f.write(json.dumps({"trajectory_id": r["trajectory_id"], "label": r["label"], "error_class": r.get("error_class"), "origin": r.get("origin"),
                                    "source": r.get("source"), "domain": r["domain"], "split": r.get("split"), "p_incorrect": float(p)}) + "\n")
    print("\nsaved", CKPT_DIR / "test_predictions.jsonl", "and", CKPT_DIR / "test_synth_predictions.jsonl")

# save per-example probabilities for Phase 7 (calibration / risk-coverage)
with open(CKPT_DIR / PRED_FILE, "w") as f:
    for r, p in zip(test_rows, probs):
        f.write(json.dumps({"trajectory_id": r["trajectory_id"], "label": r["label"], "error_class": r["error_class"],
                            "domain": r["domain"], "p_incorrect": float(p)}) + "\n")
print("\nsaved", CKPT_DIR / PRED_FILE)